In [3]:
!pip install transformers torch matplotlib tqdm

import os
import json
import glob
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaModel, RobertaConfig, RobertaTokenizer
from torch.optim import AdamW
from google.colab import drive

print(" Libraries imported successfully.")

 Libraries imported successfully.


In [4]:
drive.mount('/content/drive')

FOLDER_PATH = "/content/drive/MyDrive/dataMiningProject/CSI_Project/datasets"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f" Working on: {DEVICE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Working on: cpu


In [5]:
class MultiFileCodeDataset(Dataset):
    def __init__(self, folder_path, tokenizer, max_len=512):
        self.data = []
        self.tokenizer = tokenizer
        self.max_len = max_len

        file_pattern = os.path.join(folder_path, "*.jsonl")
        file_list = glob.glob(file_pattern)

        if not file_list:
            raise FileNotFoundError(f"can't find file in: {folder_path}")

        print(f"Combination of {len(file_list)} Files..")
        for file_path in file_list:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    try:
                        self.data.append(json.loads(line))
                    except: continue
        print(f"Total: {len(self.data)} sample")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        code_lines = item.get('lines', [])
        full_code = " ".join(code_lines)
        line_labels = item.get('label', [0] * len(code_lines))

        encoding = self.tokenizer(
            full_code, max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors="pt"
        )

        target_labels = torch.zeros(self.max_len)
        actual_labels = torch.tensor(line_labels[:self.max_len])
        target_labels[:len(actual_labels)] = actual_labels

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': target_labels
        }

In [6]:
class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2):
        super(BinaryFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets, mask):
        ins = inputs.view(-1)
        tgs = targets.view(-1).float()
        mask = mask.view(-1) == 1

        ins, tgs = ins[mask], tgs[mask]

        BCE_loss = nn.functional.binary_cross_entropy_with_logits(ins, tgs, reduction='none')
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1 - pt)**self.gamma * BCE_loss

        return torch.mean(F_loss)

class HybridVulnerabilityModel(nn.Module):
    def __init__(self, model_name="microsoft/graphcodebert-base", donor_name="claudios/VulBERTa-MLP-D2A"):
        super().__init__()
        print(f"🏗️ Initializing Hybrid Model...")

        self.config = RobertaConfig.from_pretrained(donor_name)
        self.encoder = RobertaModel.from_pretrained(donor_name)

        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        # إنتاج score لكل توكن لتحديد السطر المصاب
        logits = self.classifier(self.dropout(sequence_output)).squeeze(-1)
        return logits

print(" Model and FocalLoss defined.")

 Model and FocalLoss defined.


In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("claudios/VulBERTa-MLP-D2A")

dataset = MultiFileCodeDataset(FOLDER_PATH, tokenizer)
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)

model = HybridVulnerabilityModel(donor_name="claudios/VulBERTa-MLP-D2A").to(DEVICE)

criterion = BinaryFocalLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

print("✅ Tokenizer size matched with Model weights. Ready to train!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

Combination of 6 Files..
Total: 13109 sample
🏗️ Initializing Hybrid Model...


config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: claudios/VulBERTa-MLP-D2A
Key                        | Status     | 
---------------------------+------------+-
classifier.out_proj.weight | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Tokenizer size matched with Model weights. Ready to train!


In [ ]:
epoch_losses = []
epochs = 10

print("🚀 Starting Training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:
        optimizer.zero_grad()
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        targets = batch['labels'].to(DEVICE)

        logits = model(ids, mask)
        loss = criterion(logits, targets, mask)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    epoch_losses.append(total_loss / len(train_loader))


plt.figure(figsize=(8, 5))
plt.plot(epoch_losses, marker='o', color='b')
plt.title('Training Loss - Task 1 (Line Localization)')
plt.xlabel('Epoch')
plt.ylabel('Loss Value')
plt.grid(True)
plt.show()

print("🏁 All done! You've successfully trained the localization head.")